In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, str(Path("../../Tools/Ctrl-DNA").resolve()))
import scripts.motifs, scripts.utils

In [2]:

# =========================
# input
# =========================
offline_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_offline.tsv",
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_offline.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_offline.tsv"
]

complete_path = [
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/DNase/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HCT116/HCT116_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/HepG2/HepG2_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete.tsv", 
    "/home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete.tsv"
]


meme_path = "/home/wangyucheng/projects/SeqEval_Bench_Data/Additional_data/human/20260123061938_JASPAR2026_combined_matrices_695344.meme"
selected_ppm_path = "/home/wangyucheng/projects/SeqEval_Bench_Data/Additional_data/human/selected_ppms.csv"
MAX_N = 50000



def infer_sep(file_path: str) -> str:
    suffix = Path(file_path).suffix.lower()
    if suffix == ".tsv":
        return "\t"
    elif suffix == ".csv":
        return "\t"
    else:
        raise ValueError(f"Unsupported file type: {file_path}")


def find_sequence_column(df: pd.DataFrame) -> str:
    candidate_cols = [
        "Sequence", "sequence", "seq", "Seq", "SEQ"
    ]
    for col in candidate_cols:
        if col in df.columns:
            return col

    for col in df.columns:
        if col.lower() == "sequence":
            return col
    for col in df.columns:
        if col.lower() == "seq":
            return col

    raise ValueError(f"Cannot find sequence column. Available columns: {list(df.columns)}")


def build_dataset_labels(file_path: str):
    path = Path(file_path)

    cell_type = path.parent.name
    dataset = path.parent.parent.name

    dataset_tag = f"{dataset}_{cell_type}"

    return dataset, cell_type, dataset_tag


def load_sequences(file_path: str, max_n: int = 50000) -> pd.DataFrame:
    sep = infer_sep(file_path)
    df = pd.read_csv(file_path, sep=sep)

    seq_col = find_sequence_column(df)
    df = df.rename(columns={seq_col: "sequence"})

    df = df.dropna(subset=["sequence"]).copy()

    df["sequence"] = df["sequence"].astype(str)

    df = df[~df["sequence"].str.contains("N", na=False)].copy()

    if len(df) > max_n:
        df = df.sample(n=max_n, random_state=42)

    df = df.reset_index(drop=True)
    df.index = df.index.astype(str)

    return df




In [3]:
# =========================
# main
# =========================
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# Per-process state, populated once by _init_worker (fork start method)
_WORKER_STATE = {}


def _init_worker(meme_path, selected_ppm_path):
    """Initializer run once per worker process: load and filter motifs."""
    motifs, bg = scripts.motifs.read_meme(meme_path)
    sel = scripts.utils.load_csv(selected_ppm_path).Matrix_id.tolist()
    motifs = [m for m in motifs if m.name.decode() in sel]
    _WORKER_STATE["motifs"] = motifs
    _WORKER_STATE["bg"] = bg


def _process_one_file(file_path, split, max_n, threshold):
    """Process a single sequence file. Returns (file_path, success, message)."""
    try:
        _, _, dataset_tag = build_dataset_labels(file_path)

        # save_dir: <input file's directory> / Ctrl-DNA / offline or complete
        save_dir = Path(file_path).parent / "Ctrl-DNA" / split
        os.makedirs(save_dir, exist_ok=True)

        seqs = load_sequences(file_path, max_n=max_n)

        # motif scan
        sites = scripts.motifs.scan(
            seqs, _WORKER_STATE["motifs"], _WORKER_STATE["bg"], threshold=threshold
        )

        freq_df = pd.pivot_table(
            sites, values="start", index="SeqID", columns="Matrix_id", aggfunc="count"
        ).fillna(0)

        sites_out = os.path.join(save_dir, f"{dataset_tag}__tfbs_freq_all.csv")
        freq_df.to_csv(sites_out)

        return file_path, True, f"{len(seqs)} seqs -> {sites_out}"

    except Exception as e:
        return file_path, False, repr(e)


def run_tfbs_freq(offline_paths=None, complete_paths=None,
                  meme_path=None, selected_ppm_path=None,
                  max_n=50000, threshold=1e-4, n_workers=1):
    """
    Run TFBS motif scanning on offline / complete sequence files and
    compute per-sequence motif frequencies.

    Output location: Ctrl-DNA/offline or Ctrl-DNA/complete under the
    directory of each input file (created automatically if missing),
    named as {dataset}_{cell_type}__tfbs_freq_all.csv

    n_workers: number of parallel worker processes; 1 means sequential.
    Each worker loads the motifs only once (via the pool initializer).
    """
    jobs = [(p, "offline") for p in (offline_paths or [])]
    jobs += [(p, "complete") for p in (complete_paths or [])]
    print(f"Total {len(jobs)} jobs, n_workers = {n_workers}")

    if n_workers <= 1:
        # Sequential execution (also useful for debugging)
        _init_worker(meme_path, selected_ppm_path)
        print(f"Loaded {len(_WORKER_STATE['motifs'])} selected motifs.")
        for i, (file_path, split) in enumerate(jobs, start=1):
            print("=" * 80)
            print(f"[{i}/{len(jobs)}] Processing ({split}): {file_path}")
            _, ok, msg = _process_one_file(file_path, split, max_n, threshold)
            print(msg if ok else f"[ERROR] {msg}")
        return

    # Parallel execution: workers inherit notebook-defined functions via fork
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(
        max_workers=n_workers,
        mp_context=ctx,
        initializer=_init_worker,
        initargs=(meme_path, selected_ppm_path),
    ) as executor:
        future_to_job = {
            executor.submit(_process_one_file, file_path, split, max_n, threshold): (file_path, split)
            for file_path, split in jobs
        }
        for done, future in enumerate(as_completed(future_to_job), start=1):
            file_path, split = future_to_job[future]
            _, ok, msg = future.result()
            status = "OK" if ok else "ERROR"
            print(f"[{done}/{len(jobs)}] [{status}] ({split}) {file_path}")
            print(f"    {msg}")

In [4]:
run_tfbs_freq(
    offline_paths=offline_path,
    complete_paths=complete_path,
    meme_path=meme_path,
    selected_ppm_path=selected_ppm_path,
    max_n=MAX_N,
    n_workers=10,
)

Total 22 jobs, n_workers = 10
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs
Read 892 motifs


  0%|          | 0/603 [00:00<?, ?it/s]/tmp/ipykernel_3860293/1486666635.py:80: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=sep)
 57%|█████▋    | 344/603 [04:52<03:46,  1.14it/s]

[1/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/GM12878_offline.tsv
    27291 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/DNase/GM12878/Ctrl-DNA/offline/DNase_GM12878__tfbs_freq_all.csv


  0%|          | 0/603 [00:00<?, ?it/s] 1.16it/s]

[2/22] [OK] (offline) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/A549_offline.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/A549/Ctrl-DNA/offline/EpisomalMPRA_A549__tfbs_freq_all.csv


...

[20/22] [OK] (complete) /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/SKNSH_complete.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/EpisomalMPRA/SKNSH/Ctrl-DNA/complete/EpisomalMPRA_SKNSH__tfbs_freq_all.csv


 97%|█████████▋| 585/603 [08:12<00:14,  1.24it/s]

[21/22] [OK] (complete) /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/K562_complete.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/K562/Ctrl-DNA/complete/LentiMPRA_K562__tfbs_freq_all.csv


100%|██████████| 603/603 [08:28<00:00,  1.19it/s]


[22/22] [OK] (complete) /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/WTC11_complete.tsv
    50000 seqs -> /home/wangyucheng/projects/SeqEval_Bench_Data/LentiMPRA/WTC11/Ctrl-DNA/complete/LentiMPRA_WTC11__tfbs_freq_all.csv
